In [1]:
%pip install transformers datasets accelerate torch
%pip install scikit-learn pandas numpy nltk fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

from transformers import AutoTokenizer

# Load DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

# Two example sentences
sentences = [
    "I love this movie. It is amazing!",
    "The movie was boring and disappointing."
]

# Tokenize sentences
for sentence in sentences:

    result = tokenizer(
        sentence,
        padding="max_length",
        truncation=True,
        max_length=20
    )

    print("\nSentence:", sentence)

    print("Tokens:")
    print(tokenizer.convert_ids_to_tokens(
        result["input_ids"]
    ))

    print("Input IDs:")
    print(result["input_ids"])

    print("Attention Mask:")
    print(result["attention_mask"])


c:\python work\training\Deep Learning\week-3-assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Sentence: I love this movie. It is amazing!
Tokens:
['[CLS]', 'i', 'love', 'this', 'movie', '.', 'it', 'is', 'amazing', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Input IDs:
[101, 1045, 2293, 2023, 3185, 1012, 2009, 2003, 6429, 999, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Sentence: The movie was boring and disappointing.
Tokens:
['[CLS]', 'the', 'movie', 'was', 'boring', 'and', 'disappointing', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Input IDs:
[101, 1996, 3185, 2001, 11771, 1998, 15640, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [3]:

from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

sentences = [
    "This product is excellent and I love it.",
    "The service was terrible and disappointing.",
    "The movie was okay, but nothing special."
]

print("\n========== SENTIMENT ANALYSIS ==========")

for sentence in sentences:

    result = sentiment_pipeline(sentence)

    print("\nText:", sentence)

    print("Prediction:", result[0]["label"])

    print("Confidence:", round(result[0]["score"], 4))


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 142.44it/s]



========== SENTIMENT ANALYSIS ==========

Text: This product is excellent and I love it.
Prediction: POSITIVE
Confidence: 0.9999

Text: The service was terrible and disappointing.
Prediction: NEGATIVE
Confidence: 0.9997

Text: The movie was okay, but nothing special.
Prediction: NEGATIVE
Confidence: 0.9935


In [4]:

from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model="dslim/distilbert-NER",
    aggregation_strategy="simple"
)

sentences = [
    "Elon Musk founded companies such as Tesla and SpaceX.",
    "Google is headquartered in California.",
    "Virat Kohli plays cricket for India."
]

print("\n========== NAMED ENTITY RECOGNITION ==========")

for sentence in sentences:

    result = ner_pipeline(sentence)

    print("\nSentence:", sentence)

    for entity in result:

        print(
            "Entity:", entity["word"],
            "| Type:", entity["entity_group"],
            "| Confidence:", round(entity["score"], 4)
        )


c:\python work\training\Deep Learning\week-3-assessment\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--dslim--distilbert-NER. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\python work\training\Deep Learning\week-3-assessment\.venv\Lib\site-packages\huggin


========== NAMED ENTITY RECOGNITION ==========

Sentence: Elon Musk founded companies such as Tesla and SpaceX.
Entity: El | Type: PER | Confidence: 0.5701
Entity: ##on Musk | Type: ORG | Confidence: 0.6436
Entity: Te | Type: ORG | Confidence: 0.9985
Entity: ##sla | Type: ORG | Confidence: 0.9985
Entity: Space | Type: ORG | Confidence: 0.9981
Entity: ##X | Type: ORG | Confidence: 0.754

Sentence: Google is headquartered in California.
Entity: Google | Type: ORG | Confidence: 0.9938
Entity: California | Type: LOC | Confidence: 0.9985

Sentence: Virat Kohli plays cricket for India.
Entity: V | Type: PER | Confidence: 0.9991
Entity: ##ira | Type: PER | Confidence: 0.9991
Entity: ##t Kohli | Type: PER | Confidence: 0.9971
Entity: India | Type: LOC | Confidence: 0.9979


In [5]:

import nltk
import numpy as np
import pandas as pd
import torch
import time

from nltk.corpus import movie_reviews

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

# ------------------------------------------
# 1. Load dataset
# ------------------------------------------

nltk.download("movie_reviews")

texts = []
labels = []

for category in movie_reviews.categories():

    for file_id in movie_reviews.fileids(category):

        text = movie_reviews.raw(file_id)

        # Use short text: first 100 words
        text = " ".join(text.split()[:100])

        texts.append(text)

        # neg = 0, pos = 1
        labels.append(
            1 if category == "pos" else 0
        )

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

print("Dataset size:", len(df))

# ------------------------------------------
# 2. Train-test split
# ------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# ------------------------------------------
# 3. Load tokenizer
# ------------------------------------------

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# ------------------------------------------
# 4. Convert into Hugging Face datasets
# ------------------------------------------

train_data = Dataset.from_dict({
    "text": X_train,
    "labels": y_train
})

test_data = Dataset.from_dict({
    "text": X_test,
    "labels": y_test
})

# ------------------------------------------
# 5. Tokenization
# ------------------------------------------

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )


train_data = train_data.map(
    tokenize_function,
    batched=True
)

test_data = test_data.map(
    tokenize_function,
    batched=True
)

train_data = train_data.remove_columns(["text"])
test_data = test_data.remove_columns(["text"])

# ------------------------------------------
# 6. Load DistilBERT model
# ------------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={
        0: "NEGATIVE",
        1: "POSITIVE"
    },
    label2id={
        "NEGATIVE": 0,
        "POSITIVE": 1
    }
)

# ------------------------------------------
# 7. Evaluation metrics
# ------------------------------------------

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "f1": macro_f1
    }


# ------------------------------------------
# 8. Data collator
# ------------------------------------------

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

# ------------------------------------------
# 9. Training arguments
# ------------------------------------------

training_args = TrainingArguments(

    output_dir="./distilbert_results",

    num_train_epochs=3,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    learning_rate=2e-5,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    logging_steps=20,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=42
)

# ------------------------------------------
# 10. Create Trainer
# ------------------------------------------

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_data,

    eval_dataset=test_data,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# ------------------------------------------
# 11. Fine-tune model
# ------------------------------------------

print("\n========== DISTILBERT TRAINING ==========")

start_time = time.perf_counter()

trainer.train()

distilbert_training_time = (
    time.perf_counter() - start_time
)

print(
    "Training time:",
    round(distilbert_training_time, 2),
    "seconds"
)

# ------------------------------------------
# 12. Evaluate model
# ------------------------------------------

print("\n========== DISTILBERT EVALUATION ==========")

evaluation = trainer.evaluate()

distilbert_accuracy = evaluation["eval_accuracy"]

distilbert_f1 = evaluation["eval_f1"]

print("Accuracy:", round(distilbert_accuracy, 4))

print("Macro F1:", round(distilbert_f1, 4))

# ------------------------------------------
# 13. Save fine-tuned model
# ------------------------------------------

trainer.save_model("./saved_distilbert")

tokenizer.save_pretrained("./saved_distilbert")

print("\nModel saved successfully!")


[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


Dataset size: 2000


Loading weights: 100%|██████████| 100/100 [00:01<00:00, 99.31it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



========== DISTILBERT TRAINING ==========


c:\python work\training\Deep Learning\week-3-assessment\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.647886,0.624044,0.630000,0.629073
2,0.452182,0.646959,0.677500,0.665749
3,0.348832,0.643587,0.667500,0.666196


... (3 outputs left out)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.348832,0.643587,3,0.667500,0.666196


Accuracy: 0.6675
Macro F1: 0.6662


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.79it/s]



Model saved successfully!


In [6]:

# ==========================================
# TF-IDF VS DISTILBERT COMPARISON
# ==========================================

import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------
# 1. Train TF-IDF model
# ------------------------------------------

tfidf_model = Pipeline([

    (
        "tfidf",
        TfidfVectorizer(
            max_features=10000,
            stop_words="english"
        )
    ),

    (
        "classifier",
        LogisticRegression(max_iter=1000)
    )

])

start_time = time.perf_counter()

tfidf_model.fit(X_train, y_train)

tfidf_training_time = (
    time.perf_counter() - start_time
)

# ------------------------------------------
# 2. TF-IDF predictions and inference time
# ------------------------------------------

start_time = time.perf_counter()

tfidf_predictions = tfidf_model.predict(X_test)

tfidf_inference_time = (
    time.perf_counter() - start_time
) / len(X_test) * 1000

tfidf_accuracy = accuracy_score(
    y_test,
    tfidf_predictions
)

tfidf_f1 = f1_score(
    y_test,
    tfidf_predictions,
    average="macro"
)

# ------------------------------------------
# 3. DistilBERT predictions
# ------------------------------------------

start_time = time.perf_counter()

distilbert_predictions = trainer.predict(test_data)

distilbert_inference_time = (
    time.perf_counter() - start_time
) / len(X_test) * 1000

bert_logits = distilbert_predictions.predictions

bert_predictions = np.argmax(
    bert_logits,
    axis=1
)

distilbert_accuracy = accuracy_score(
    y_test,
    bert_predictions
)

distilbert_f1 = f1_score(
    y_test,
    bert_predictions,
    average="macro"
)

# ------------------------------------------
# 4. Comparison table
# ------------------------------------------

comparison = pd.DataFrame({

    "Model": [
        "TF-IDF + Logistic Regression",
        "DistilBERT"
    ],

    "Accuracy": [
        tfidf_accuracy,
        distilbert_accuracy
    ],

    "Macro F1": [
        tfidf_f1,
        distilbert_f1
    ],

    "Training Time (seconds)": [
        tfidf_training_time,
        distilbert_training_time
    ],

    "Inference Time (ms/text)": [
        tfidf_inference_time,
        distilbert_inference_time
    ]

})

print("\n========== FINAL MODEL COMPARISON ==========")

print(
    comparison.round(4).to_string(index=False)
)

# Save comparison table
comparison.to_csv(
    "model_comparison.csv",
    index=False
)

print("\nComparison saved to model_comparison.csv")


c:\python work\training\Deep Learning\week-3-assessment\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



========== FINAL MODEL COMPARISON ==========
                       Model  Accuracy  Macro F1  Training Time (seconds)  Inference Time (ms/text)
TF-IDF + Logistic Regression    0.6425    0.6423                   0.8620                    0.1321
                  DistilBERT    0.6675    0.6662                3497.3004                  197.3039

Comparison saved to model_comparison.csv
